#  Phase 3: Advanced LangChain — AI Research Assistant (Gemini)

**Objective:** Build a context-aware AI Research Assistant using LangChain Agents, Tools, and Memory.

| Component | Technology |
|-----------|------------|
| LLM | Google Gemini 1.5 Pro |
| Agent | ReAct / Zero-shot |
| Tools | Google Search, Wikipedia, Python REPL, SQL |
| Memory | Buffer, Entity, Vector (Chroma) |
| Storage | SQLite (conversation history) |

---

## Section 1: Installation & Setup

In [ ]:
# Install all required packages
!pip install -q langchain langchain-google-genai langchain-community langchain-experimental
!pip install -q google-generativeai
!pip install -q google-search-results  # SerpAPI
!pip install -q wikipedia
!pip install -q chromadb
!pip install -q sqlalchemy
!pip install -q faiss-cpu
!pip install -q python-dotenv
!pip install -q tabulate

print("All packages installed successfully!")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  

 
os.environ["GOOGLE_API_KEY"]   = os.getenv("GOOGLE_API_KEY",  "key")
os.environ["SERPAPI_API_KEY"]  = os.getenv("SERPAPI_API_KEY", "key")


# Validate
for var in ["GOOGLE_API_KEY", "SERPAPI_API_KEY"]:
    val = os.environ.get(var, "")
    masked = val[:8] + "..." if len(val) > 8 else "❌ NOT SET"
    print(f"{var}: {masked}")

print("\nEnvironment configured!")

In [ ]:
# Core imports
import warnings, json, sqlite3, datetime
warnings.filterwarnings("ignore")

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
# langchain.schema was removed in LangChain >=0.2 — import from langchain_core instead
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from IPython.display import display, Markdown

# ── Shared LLM instance (
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    max_output_tokens=1500,
)

def md(text): display(Markdown(text))   

print("LLM initialised:", llm.model)

---
## Section 2: LangChain Memory

Memory lets your chatbot *remember* past interactions within or across sessions.

| Memory Type | What it remembers | Best for |
|---|---|---|
| `ConversationBufferMemory` | Full raw transcript | Short chats |
| `ConversationBufferWindowMemory` | Last *k* turns | Medium chats |
| `ConversationSummaryMemory` | Running summary | Long chats |
| `EntityMemory` | Named entities (people, places) | Personal assistants |
| `VectorStoreRetrieverMemory` | Semantic similarity search | Knowledge bases |

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain
from langchain_core.messages import HumanMessage


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

# 2. Setup Memory Structure via classic backport
buffer_memory = ConversationBufferMemory(return_messages=True)

# 3. Create the multi-turn chain wrapper
buffer_chain = ConversationChain(
    llm=llm,
    memory=buffer_memory,
    verbose=False,
)

# 4. Multi-turn chat simulation arrays
turns = [
    "Hi! My name is Ali and I'm researching quantum computing.",
    "What's the most exciting recent development in this field?",
    "Can you remind me what topic I said I was researching?",
]

print("=" * 60)
print("ConversationBufferMemory Demo (Powered by Gemini)")
print("=" * 60)

for turn in turns:
    response = buffer_chain.predict(input=turn)
    print(f"\n👤 User : {turn}")
    print(f"🤖 Bot  : {response}")

print("\n" + "=" * 60)
print("Raw memory buffer history:")
print("=" * 60)
for msg in buffer_memory.chat_memory.messages:
    role = "👤" if isinstance(msg, HumanMessage) else "🤖"
    print(f"  {role} {msg.content[:80]}..." if len(msg.content) > 80 else f"  {role} {msg.content}")


In [ ]:
# ── 2b. ConversationSummaryMemory — handles long conversations ───────────────

from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.messages import HumanMessage

# Initialize summary memory 
summary_memory = ConversationSummaryMemory(llm=llm, return_messages=True)


summary_memory.save_context(
    {"input": "Hi, I am planning a 10-day trip to Japan focusing on food and history."},
    {"output": "That sounds amazing! You should visit Tokyo, Kyoto, and Osaka. I can recommend some historic ramen shops."}
)
summary_memory.save_context(
    {"input": "I prefer traditional architecture over modern skyscrapers."},
    {"output": "In that case, we should dedicate more days to Kyoto and Nara to see ancient temples."}
)

print("=" * 60)
print("ConversationSummaryMemory Demo")
print("=" * 60)

# Check what the memory thinks the history looks like now!
# It will automatically generate a short paragraph summarizing everything said so far.
print("\n📋 Current Memory Summary:")
moving_summary = summary_memory.load_memory_variables({})
print(moving_summary.get("history", moving_summary))


In [ ]:
# ── 2c. Vector-based Memory (Chroma) — semantic long-term memory ─────────────
from langchain_classic.memory import VectorStoreRetrieverMemory
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage

# 1. Initialize Google's modern embedding model using the working API path
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 2. Create an in-memory Chroma database to hold the memories vectors
vectorstore = Chroma.from_texts(
    texts=["initialization"],
    embedding=embeddings
)

# 3. Create the retriever (search engine) for our memory bank
# search_kwargs={"k": 1} means "find the single most relevant memory"
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
vector_memory = VectorStoreRetrieverMemory(retriever=retriever)

# 4. Save some completely unrelated facts into this AI's brain
vector_memory.save_context({"input": "My favorite color is electric blue."}, {"output": "Got it."})
vector_memory.save_context({"input": "I have a golden retriever named Sparky."}, {"output": "Logged."})
vector_memory.save_context({"input": "I am allergic to peanuts."}, {"output": "Stay away from peanuts!"})

print("=" * 60)
print("VectorStoreRetrieverMemory Demo")
print("=" * 60)

# 5. Ask a question that requires searching the memory bank semantics
query = "What kind of pet do I have?"
contextual_memories = vector_memory.load_memory_variables({"prompt": query})

print(f"🔍 Searching memory for: '{query}'")
print("-" * 60)
print("📋 Relevant Memory Retrieved:")
print(contextual_memories.get("history"))

---
##  Section 3: LangChain Tools

Tools give agents the ability to act on the world — search the web, run code, query databases, and more.

In [ ]:
# ── 3a. Google Search via SerpAPI ────────────────────────────────────────────
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import Tool 

# 1. Initialize the search wrapper (uses the SERPAPI_API_KEY you saved earlier)
search = SerpAPIWrapper()

# 2. Wrap it inside a LangChain Tool object so an Agent can use it later
search_tool = Tool(
    name="Current Search",
    func=search.run,
    description="Useful for when you need to answer questions about current events or real-time web data."
)

print("=" * 60)
print("SerpAPI Google Search Tool Demo")
print("=" * 60)

# 3. Test the search functionality
query = "What is the latest major breakthrough in quantum computing?"
print(f"🌐 Triggering live Google search for: '{query}'...\n")

try:
    result = search_tool.run(query)
    print("📋 Search Result Snippet:")
    print("-" * 60)
    print(result)
except Exception as e:
    print(f"Search failed. Check your SerpAPI key. Error: {e}")

In [ ]:
!pip install wikipedia -q

In [ ]:

# 3b. Wikipedia Tool (With User-Agent Rate-Limit Fix)

import wikipedia
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

#  Explicitly set a custom User-Agent to prevent Wikimedia from blocking the request
wikipedia.set_user_agent("TechnicalCourseworkBot/1.0 (contact: student@example.com)")

# Initialize the Wikipedia tool components with your custom specs
wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1000)
)

print("=" * 60)
print("Wikipedia Tool Test")
print("=" * 60)

# Run the query against Wikipedia's database
try:
    result = wiki_tool.run("Transformer neural network architecture")
    print(result[:600] + " ...")
except Exception as e:
    print(f"Wikipedia fetch failed. Error: {e}")

In [ ]:
# ── 3c. Python REPL Tool — agent can write & run code ────────────────────────
from langchain_experimental.tools import PythonREPLTool

python_tool = PythonREPLTool()

# Let the tool execute Python for computations
code = """
import math
# Calculate compound interest
principal = 10000
rate = 0.07
years = 10
result = principal * (1 + rate) ** years
print(f'$10,000 at 7% for 10 years = ${result:,.2f}')
"""
print("=" * 60)
print("Python REPL Tool Test")
print("=" * 60)
print(python_tool.run(code))

In [ ]:
# ── 3d. SQLite Database Tool ─────────────────────────────────────────────────
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
import sqlite3

# Create a research database
DB_PATH = "research_assistant.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS research_sessions;
DROP TABLE IF EXISTS research_topics;
DROP TABLE IF EXISTS conversation_log;

CREATE TABLE research_sessions (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT NOT NULL,
    created_at TEXT DEFAULT (datetime('now')),
    user_name  TEXT
);

CREATE TABLE research_topics (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    topic      TEXT NOT NULL,
    category   TEXT,
    relevance  REAL DEFAULT 1.0,
    created_at TEXT DEFAULT (datetime('now'))
);

CREATE TABLE conversation_log (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT,
    role       TEXT CHECK(role IN ('user','assistant')),
    message    TEXT,
    timestamp  TEXT DEFAULT (datetime('now'))
);
""")

# Seed some topics
topics = [
    ("Quantum Computing",   "Physics",      0.95),
    ("Large Language Models","AI/ML",       0.98),
    ("CRISPR Gene Editing", "Biotechnology",0.90),
    ("Climate Change",       "Environment", 0.88),
    ("Blockchain",           "Technology",  0.75),
]
cur.executemany(
    "INSERT INTO research_topics (topic, category, relevance) VALUES (?, ?, ?)",
    topics,
)
conn.commit()
conn.close()

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")
print("SQLite database created:", DB_PATH)
print("Tables:", db.get_table_names())

In [ ]:
import sqlite3
import pandas as pd

# 1. Open a connection to your assistant's database
conn = sqlite3.connect("research_assistant.db")

# 2. Write a clean query matching your schema columns
query = """
SELECT id, session_id, role, message, timestamp
FROM conversation_log
ORDER BY timestamp DESC
LIMIT 10;
"""

# 3. Read the SQL results directly into a Pandas DataFrame
df = pd.read_sql_query(query, conn)

# 4. Close the connection
conn.close()

# 5. Display the data frame cleanly in your Colab output window
pd.set_option('display.max_colwidth', None) 
df

In [ ]:
# Custom SQL lookup tool (safe read-only queries)
def query_research_db(query: str) -> str:
    """Query the research topics database. Input should be a plain-English question about topics."""
    conn = sqlite3.connect(DB_PATH)
    cur  = conn.cursor()
    try:
        # Map natural language to SQL
        q = query.lower()
        if "top" in q or "highest" in q or "best" in q:
            cur.execute("SELECT topic, category, relevance FROM research_topics ORDER BY relevance DESC LIMIT 3")
        elif "categor" in q:
            cur.execute("SELECT DISTINCT category FROM research_topics")
        else:
            cur.execute("SELECT topic, category, relevance FROM research_topics")
        rows = cur.fetchall()
        conn.close()
        if not rows:
            return "No results found."
        return "\n".join([str(r) for r in rows])
    except Exception as e:
        conn.close()
        return f"DB error: {e}"

db_tool = Tool(
    name="Research Database",
    description="Query the local research topics database to find stored topics, categories, and relevance scores.",
    func=query_research_db,
)

print("DB Tool test:")
print(query_research_db("top research topics"))

---
## = Section 4: LangChain Agents

An **Agent** uses an LLM as a reasoning engine to decide which tools to call and in what order.

```
User Input → Agent (Reasoning Loop) → Tool Call → Observation → … → Final Answer
```

### Agent Types
| Type | Description |
|------|-------------|
| **Zero-shot ReAct** | Uses tool descriptions alone to decide; no examples needed |
| **Structured Chat** | Handles multi-input tools; good for complex tasks |
| **OpenAI Functions** | Uses native OpenAI function-calling for reliability |

In [ ]:
# ── 4a. Modern LangChain Agent (Sanitized Tool Names) ────────────────────────
from langchain.agents import create_agent
from langchain_core.tools import Tool
from langchain_experimental.utilities import PythonREPL

# 1. Sanitize and override names of previous tools to satisfy Gemini's API syntax
search_tool.name = "google_search"
wiki_tool.name = "wikipedia_lookup"


repl_instance = PythonREPL()
python_repl_tool = Tool(
    name="python_calc",
    description="A Python shell. Use this to execute python commands for accurate math calculations and logic.",
    func=repl_instance.run
)

# 3. Gather the freshly sanitized tools into the roster
tools = [search_tool, wiki_tool, python_repl_tool]

# 4. Compile the modern agent using your Gemini model and the clean tools list
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are an autonomous research assistant. Use your tools dynamically "
        "to verify and process facts. Always provide a thoroughly calculated, factual response."
    )
)

print("=" * 60)
print("🤖 Sanitized Autonomous Agent Compiled Successfully!")
print("=" * 60)

# 5. Kick off the complex query
test_query = "Who won the most recent Super Bowl, and what is that number multiplied by 5?"
print(f"User Query: {test_query}\n")

# Run the agent loop!
try:
    response = agent.invoke({
        "messages": [
            {"role": "user", "content": test_query}
        ]
    })

    print("📋 Agent Final Output:")
    print("-" * 60)
    print(response["messages"][-1].content)

except Exception as e:
    print(f" Execution failed: {e}")

In [ ]:

# ReAct Agent — Multi-step Research Query
from IPython.display import display, Markdown

query = (
    "Search for the latest developments in quantum computing in 2024, "
    "then check what Wikipedia says about quantum entanglement, "
    "and finally calculate how many qubits would be needed if the number "
    "doubles every 18 months starting from 100 qubits over 5 years."
)

print("=" * 60)
print("Launching Multi-Step Autonomous Agent Loop...")
print("=" * 60)

try:
    # Invoke the modern agent using the corrected message dictionary schema
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": query}
        ]
    })

    print("\n" + "=" * 60)
    print("🏆 FINAL ANSWER:")
    print("=" * 60)

    # Extract the modern message content payload safely
    final_output = result["messages"][-1].content

    # If the response is wrapped in the Google GenAI metadata list format, extract the text
    if isinstance(final_output, list) and len(final_output) > 0 and 'text' in final_output[0]:
        final_markdown = final_output[0]['text']
    else:
        final_markdown = final_output

    # Render the final output beautifully using Markdown
    display(Markdown(final_markdown))

except NameError:
    print("Error: The 'agent' variable isn't initialized yet. Please re-run your compilation cell above first!")
except Exception as e:
    print(f"Execution failed: {e}")

In [ ]:

# ── 4b. Modern Tool-Calling Agent (Debugged Version) ───────────────────

from langchain.agents import create_agent
from langchain_core.tools import Tool
from langchain_experimental.utilities import PythonREPL
from pprint import pprint


# Verify Tool Names

search_tool.name = "google_search"
wiki_tool.name = "wikipedia_lookup"
# Python Calculator Tool
repl_instance = PythonREPL()

python_repl_tool = Tool(
    name="python_calc",
    description=(
        "A Python shell. Use this tool for mathematical calculations, "
        "logic, formulas, and data processing."
    ),
    func=repl_instance.run
)

# ------------------------------------------------------------------
# Tool List
# ------------------------------------------------------------------

tools = [
    search_tool,
    wiki_tool,
    python_repl_tool
]


# Create Agent
functions_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are an expert AI Research Assistant.

    Your responsibilities:
    1. Search for current information when needed.
    2. Use Wikipedia for background knowledge.
    3. Use Python for calculations.
    4. Combine all findings into one final answer.
    5. Explain your reasoning clearly.
    """
)

print("=" * 60)
print(" Modern Tool-Calling Agent Compiled Successfully!")
print("=" * 60)


# Complex Query


query = (
    "Search for the latest developments in quantum computing in 2024, "
    "then check what Wikipedia says about quantum entanglement, "
    "and finally calculate how many qubits would be needed if the number "
    "doubles every 18 months starting from 100 qubits over 5 years."
)

print(f"\n Launching Query:\n{query}\n")

#execte agent

try:

    response = functions_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    print("\n" + "=" * 60)
    print("📋 RAW RESPONSE")
    print("=" * 60)

    pprint(response)

    print("\n" + "=" * 60)
    print("AGENT FINAL OUTPUT")
    print("=" * 60)

    if "messages" in response:

        messages = response["messages"]

        print(f"\nTotal Messages Returned: {len(messages)}")

        last_message = messages[-1]

        print("\nLast Message Object:")
        print(last_message)

        if hasattr(last_message, "content"):

            final_content = last_message.content

            print("\nFinal Content:\n")

            if isinstance(final_content, str):
                print(final_content)

            elif isinstance(final_content, list):

                for item in final_content:

                    if isinstance(item, dict):
                        print(item.get("text", item))

                    else:
                        print(item)

            else:
                print(final_content)

        else:
            print("⚠️ No content attribute found in last message.")

    else:
        print("⚠️ No messages key found in response.")
        pprint(response)

except Exception as e:

    print("\nAGENT EXECUTION FAILED")
    print(type(e).__name__)
    print(str(e))


---
## Section 5: Building the Full AI Research Assistant

Now we combine **Agents + Memory + Database** into a complete, production-ready assistant.

In [ ]:
import uuid
import sqlite3
from langchain.agents import create_agent
from langchain_core.tools import Tool
from langchain_experimental.utilities import PythonREPL

class ResearchAssistant:
    def __init__(self, model_instance, search_tool, wiki_tool, db_path="research_assistant.db"):
        self.llm = model_instance
        self.db_path = db_path

        # 1. Sanitize tool naming formats for the Gemini API
        search_tool.name = "google_search"
        wiki_tool.name = "wikipedia_lookup"

        repl_instance = PythonREPL()
        python_repl_tool = Tool(
            name="python_calc",
            description="A Python shell. Use this to execute python commands for accurate math calculations.",
            func=repl_instance.run
        )

        self.tools = [search_tool, wiki_tool, python_repl_tool]

        # 2. Compile our core underlying engine agent
        self.agent = create_agent(
            model=self.llm,
            tools=self.tools,
            system_prompt=(
                "You are an expert research assistant. Analyze the user query, "
                "break it down into clear execution steps, and use your tools efficiently. "
                "Always look back at the conversation history provided below to retain context."
            )
        )

    def _get_history_from_db(self, session_id):
        """Helper to fetch past chat messages from your SQLite table to rebuild context."""
        messages = []
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()
            # Fetch the last 6 messages (3 turns) to act as a sliding window memory
            cursor.execute(
                "SELECT role, message FROM conversation_log WHERE session_id = ? ORDER BY id DESC LIMIT 6",
                (session_id,)
            )
            rows = cursor.fetchall()
            conn.close()

            # Reverse them to keep them in chronological order
            for role, text in reversed(rows):
                messages.append({"role": role, "content": text})
        except Exception:
            pass # Fallback if database table is empty or uninitialized
        return messages

    def _save_to_db(self, session_id, role, message):
        """Helper to write new conversational messages into the local SQLite log table."""
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()
            cursor.execute(
                "INSERT INTO conversation_log (session_id, role, message) VALUES (?, ?, ?)",
                (session_id, role, str(message))
            )
            conn.commit()
            conn.close()
        except Exception as e:
            print(f"⚠️ Database logging warning: {e}")

    def run_session(self, query, session_id=None):
        """Executes a full research iteration step with live local disk logging."""
        if not session_id:
            session_id = str(uuid.uuid4())

        print(f"📁 Session ID: {session_id}")

        # Rebuild sliding history from your SQLite tables
        history = self._get_history_from_db(session_id)

        # Append the new incoming user question
        history.append({"role": "user", "content": query})

        # Log user query to disk database
        self._save_to_db(session_id, "user", query)

        try:
            # Invoke agent engine
            response = self.agent.invoke({"messages": history})

            # Extract final answer content safely
            final_output = response["messages"][-1].content
            if isinstance(final_output, list) and len(final_output) > 0 and 'text' in final_output[0]:
                clean_text = final_output[0]['text']
            else:
                clean_text = final_output

            # Log assistant response back to disk database
            self._save_to_db(session_id, "assistant", clean_text)
            return clean_text, session_id

        except Exception as e:
            return f" Execution failed: {e}", session_id
    def research(self, query, session_id=None):
        answer, session_id = self.run_session(query,
                                             session_id=session_id)
        return answer

print("=" * 60)
print("🏗️ ResearchAssistant Class Built Successfully!")
print("=" * 60)

In [ ]:
# ── Instantiate the assistant ────────────────────────────────────────────────
#assistant = ResearchAssistant(user_name="Ali")
# Instantiate Research Assistant

assistant = ResearchAssistant(
    model_instance=llm,
    search_tool=search_tool,
    wiki_tool=wiki_tool
)

print(" Research Assistant Created Successfully!")

---
##  Section 6: Live Research Sessions

Let's run a realistic multi-turn research conversation.

In [ ]:
# ── Turn 1: Initial research question ────────────────────────────────────────
print("=" * 65)
print("RESEARCH SESSION — AI & Quantum Computing")
print("=" * 65)

q1 = "What are the latest breakthroughs in quantum computing in 2024?"
print(f"\n👤 Ali: {q1}\n")

r1 = assistant.research(q1)
md(r1)

In [ ]:
# ── Turn 2: Follow-up that requires memory ───────────────────────────────────
q2 = "How do these quantum advances compare to progress in classical AI/ML?"
print(f"\n👤 Ali: {q2}\n")

r2 = assistant.research(q2)
md(r2)

In [ ]:
# ── Turn 3: Computation via Python REPL ─────────────────────────────────────
q3 = (
    "If a quantum computer currently processes 1000 operations per second "
    "and speed doubles every 2 years (quantum Moore's law), "
    "how fast will it be in 10 years? Show the year-by-year progression."
)
print(f"\n👤 Ali: {q3}\n")

r3 = assistant.research(q3)
md(r3)

In [ ]:
# ── Turn 4: Database lookup ──────────────────────────────────────────────────
q4 = "Check the research database and tell me which topics I should explore next based on what's stored."
print(f"\n👤 Ali: {q4}\n")

r4 = assistant.research(q4)
md(r4)

In [ ]:
# ── Turn 5: Memory test — refers to earlier turns ────────────────────────────
q5 = "Summarise everything we've discussed in this session so far as bullet points."
print(f"\n Ali: {q5}\n")

r5 = assistant.research(q5)
md(r5)

---
## Section 7: Session Analytics & Persistence

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("research_assistant.db")

df = pd.read_sql_query(
    "SELECT * FROM conversation_log",
    conn
)

conn.close()

df

In [ ]:
import sqlite3
import pandas as pd

print("=" * 60)
print("SQLITE PERSISTED CONVERSATION LOG")
print("=" * 60)

conn = sqlite3.connect("research_assistant.db")

df = pd.read_sql_query(
    "SELECT * FROM conversation_log",
    conn
)

conn.close()

display(df)

In [ ]:
# ── Session statistics ───────────────────────────────────────────────────────
import sqlite3
import pandas as pd

conn = sqlite3.connect("research_assistant.db")

df = pd.read_sql_query(
    "SELECT * FROM conversation_log",
    conn
)

conn.close()

print("=" * 60)
print("SESSION STATISTICS")
print("=" * 60)

print("Total Messages:", len(df))
print("User Messages:", len(df[df["role"] == "user"]))
print("Assistant Messages:", len(df[df["role"] == "assistant"]))
print("Unique Sessions:", df["session_id"].nunique())

In [ ]:
# ── Visualise session activity ────────────────────────────────────────────────
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Load data from SQLite

conn = sqlite3.connect("research_assistant.db")

df = pd.read_sql_query(
    "SELECT role, message FROM conversation_log",
    conn
)

conn.close()

# Calculate message lengths

df["length"] = df["message"].astype(str).apply(len)

# Plot

plt.figure(figsize=(10, 5))

colors = [
    "blue" if role == "user" else "green"
    for role in df["role"]
]

plt.bar(
    range(len(df)),
    df["length"],
    color=colors
)

plt.title("Conversation Message Lengths")
plt.xlabel("Message Number")
plt.ylabel("Characters")

plt.show()

---
## Section 8: Interactive Chat Interface

In [ ]:
# ── Programmatic multi-turn demo ──────────────────────────────

demo_queries = [
    "What are the key differences between supervised and unsupervised learning?",
    "Calculate the training time if I have 1M samples and processing takes 0.001 seconds each, with 10 epochs.",
    "Based on our conversation, what should I research next?"
]

new_assistant = ResearchAssistant(
    model_instance=llm,
    search_tool=search_tool,
    wiki_tool=wiki_tool
)

session_id = None

for q in demo_queries:

    print("\n" + "─" * 60)
    print(f"👤 User: {q}")
    print("─" * 60)

    response, session_id = new_assistant.run_session(
        query=q,
        session_id=session_id
    )

    print("🤖 Assistant:")
    print(response)

print("\nDemo session complete.")
print("📁 Session ID:", session_id)

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("research_assistant.db")

df = pd.read_sql_query(
    "SELECT * FROM conversation_log",
    conn
)

conn.close()

print("=" * 60)
print("SESSION STATISTICS")
print("=" * 60)

print("Total Messages:", len(df))
print("User Messages:", len(df[df['role'] == 'user']))
print("Assistant Messages:", len(df[df['role'] == 'assistant']))
print("Unique Sessions:", df['session_id'].nunique())

In [ ]:
# ── OPTIONAL: True interactive loop ─────────────────────────────

interactive_assistant = ResearchAssistant(
    model_instance=llm,
    search_tool=search_tool,
    wiki_tool=wiki_tool
)

session_id = None

print("🤖 AI Research Assistant")
print("Type 'quit' to exit")
print("-" * 60)

while True:

    user_input = input("\n👤 You: ").strip()

    if not user_input:
        continue

    if user_input.lower() == "quit":
        print("\n👋 Goodbye!")
        break

    response, session_id = interactive_assistant.run_session(
        query=user_input,
        session_id=session_id
    )

    print("\n🤖 Assistant:")
    print(response)

---
##  Section 9: Cross-Session Memory (Persistent Vector Store)

In [ ]:
# ── Save session knowledge to ChromaDB for future sessions ───────────────────
from langchain_community.vectorstores import Chroma

CHROMA_DIR = "./chroma_research_db"

def save_session_to_vectorstore(assistant_instance, persist_dir: str = CHROMA_DIR):
    """Extract assistant responses and embed them for cross-session retrieval."""
    history = assistant_instance.get_session_history()
    docs    = [
        f"[{ts}] {role}: {msg}"
        for role, msg, ts in history
        if role == "assistant" and len(msg) > 50
    ]
    if not docs:
        print("No assistant messages to save.")
        return None

    vs = Chroma.from_texts(
        texts=docs,
        embedding=GoogleGenerativeAIEmbeddings(model='models/embedding-001'),
        persist_directory=persist_dir,
        collection_name="research_history",
    )
    print(f" Saved {len(docs)} messages to vector store at '{persist_dir}'")
    return vs

def load_relevant_history(query: str, persist_dir: str = CHROMA_DIR, k: int = 3) -> str:
    """Retrieve the most relevant past answers for a new query."""
    vs = Chroma(
        persist_directory=persist_dir,
        embedding_function=GoogleGenerativeAIEmbeddings(model='models/embedding-001'),
        collection_name="research_history",
    )
    results = vs.similarity_search(query, k=k)
    return "\n\n".join([f"📌 {r.page_content[:200]}" for r in results])

# Save the current session
vs = save_session_to_vectorstore(assistant)

# Retrieve relevant history for a new query
print("\n🔍 Relevant past research for: 'quantum AI hybrid systems'")
print("-" * 60)
print(load_relevant_history("quantum AI hybrid systems"))

---
## Section 10: Summary & Architecture Diagram

In [ ]:
summary = """
## 🏁 Phase 3 Complete — What You Built

### Architecture
```
User Query
    │
    ▼
┌──────────────────────────────────────────────────────┐
│              ResearchAssistant                        │
│                                                      │
│  ┌──────────┐   ┌─────────────────────────────────┐  │
│  │  Memory  │   │       OpenAI Functions Agent    │  │
│  │ (Window  │◄──│  Thought → Action → Observation │  │
│  │  k=6)    │   └────────────┬────────────────────┘  │
│  └──────────┘                │ selects tool           │
│                    ┌─────────┼─────────┐             │
│               Google   Wikipedia  Python  SQLite     │
│               Search    Tool      REPL    DB Tool    │
│                                                      │
│  ┌──────────────────────────────────────────────────┐ │
│  │            SQLite Persistence Layer              │ │
│  │  sessions | conversation_log | research_topics  │ │
│  └──────────────────────────────────────────────────┘ │
│  ┌──────────────────────────────────────────────────┐ │
│  │        ChromaDB (Cross-session Vector Memory)   │ │
│  └──────────────────────────────────────────────────┘ │
└──────────────────────────────────────────────────────┘
    │
    ▼
Structured Response + Updated Memory + DB Log
```

### Key Concepts Covered
| Concept | Implementation |
|---------|---------------|
| **ReAct Agent** | Thought-Action-Observation reasoning loop |
| **Functions Agent** | OpenAI native tool calling |
| **Buffer Memory** | Sliding window of last 6 turns |
| **Summary Memory** | Auto-compresses long conversations |
| **Vector Memory** | Semantic search over past sessions (Chroma) |
| **Google Search** | SerpAPI real-time web search |
| **Wikipedia** | Structured knowledge retrieval |
| **Python REPL** | Agent-driven code execution |
| **SQLite** | Persistent conversation + topic storage |


"""

md(summary)